# VORTEX-AI: full-scale neural baselines on a GPU

1. Turn the GPU on (Kaggle: Settings > Accelerator > GPU T4. Colab: Runtime > Change runtime type > T4 GPU).
2. Provide `vortex_bundle.zip` (Kaggle: Add Input > Upload > New Dataset. Colab: the upload cell below asks for it).
3. Run all cells. When it finishes, download `fullscale_results.zip` and put it in the project folder on the laptop.

If a session ends before everything is done, start again and also provide the `fullscale_results.zip` you already have: finished runs are skipped.

In [ ]:
import glob, os, shutil, zipfile
import torch
print('GPU available:', torch.cuda.is_available(), torch.cuda.get_device_name(0) if torch.cuda.is_available() else '')
assert torch.cuda.is_available(), 'Turn the GPU on first (see step 1).'

KAGGLE = os.path.isdir('/kaggle/working')
WORK = '/kaggle/working' if KAGGLE else '/content'
os.chdir(WORK)

def find(name):
    hits = glob.glob('/kaggle/input/**/' + name, recursive=True) + glob.glob(os.path.join(WORK, name))
    return hits[0] if hits else None

if not os.path.isdir(os.path.join(WORK, 'vortex', 'crisisbench')):
    z = find('vortex_bundle.zip')
    unz = glob.glob('/kaggle/input/**/vortex/crisisbench', recursive=True)   # Kaggle unpacks uploaded zips
    if z:
        zipfile.ZipFile(z).extractall(WORK)
    elif unz:
        shutil.copytree(os.path.dirname(unz[0]), os.path.join(WORK, 'vortex'), dirs_exist_ok=True)
    else:
        from google.colab import files          # Colab: choose vortex_bundle.zip (and fullscale_results.zip if you have one)
        for name in files.upload():
            if name.startswith('vortex_bundle'):
                zipfile.ZipFile(name).extractall(WORK)

# earlier partial results, if any: finished runs are then skipped
prev = find('fullscale_results.zip')
if prev:
    zipfile.ZipFile(prev).extractall(os.path.join(WORK, 'vortex'))
prev_dir = glob.glob('/kaggle/input/**/results/runs', recursive=True)
if prev_dir:
    shutil.copytree(os.path.dirname(os.path.dirname(prev_dir[0])), os.path.join(WORK, 'vortex'), dirs_exist_ok=True)
print(sorted(os.listdir(os.path.join(WORK, 'vortex'))))

In [ ]:
os.chdir(os.path.join(WORK, 'vortex'))
os.environ['CRISISBENCH_DEVICE'] = 'cuda'
os.environ['PYTHONWARNINGS'] = 'ignore'
HOURS = 11 if KAGGLE else 3.5      # stop starting new runs before the session limit
!python -m crisisbench.fullscale --hours {HOURS} --out {WORK}/fullscale_results.zip

In [ ]:
print(os.path.getsize(f'{WORK}/fullscale_results.zip') / 1e6, 'MB')
if not KAGGLE:
    from google.colab import files
    files.download(f'{WORK}/fullscale_results.zip')